<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drexelwireless/ece512-courseware/blob/main/notebooks/lectures/week02-pathloss-interference-linkbudget.ipynb)

# Week 2 — Path Loss, Interference, and Link Budgets
**ECE 512 Wireless Communications** · companion notebook to the Week 2 lecture

This notebook lets you experiment with the ideas from the slides:

1. Log-distance path loss and lognormal shadowing (fitting $\beta$ and $\sigma$ to measurements)
2. Co-channel interference: downlink SIR $\Lambda$ vs. cluster size $N$ and path-loss exponent $\beta$
3. A map of $\Lambda$ over the cell
4. Cell sectoring ($120^\circ$ and $60^\circ$)
5. Thermal noise $kTB$ and noise figure
6. Link budget calculator (28 GHz mmWave case study)
7. Shadow margin, area outage, handoff gain, and radio coverage

Run the setup cell first. In Colab: *Runtime → Run all*.

In [ ]:
# --- Setup: installs the course package in Google Colab (safe to re-run) ---
COURSEWARE_URL = "git+https://github.com/drexelwireless/ece512-courseware"
import importlib.util
if importlib.util.find_spec("ece512") is None:
    get_ipython().run_line_magic("pip", f"install -q {COURSEWARE_URL}")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
from ece512 import cells, pathloss as pl
from ece512.units import linear_to_db, db_to_linear
from ece512.hexgrid import draw_cell, hex_vertices

rng = np.random.default_rng(512)

## 1. Log-distance path loss and lognormal shadowing
Free space gives $P_{\mathrm{RX}} \propto d^{-2}$. Real environments are fitted with a **path-loss exponent** $\beta$
plus a random **shadowing** term:
$$P_{\mathrm{RX}|\mathrm{dB}}(d) = \mathrm{E}\left[P_{\mathrm{RX}|\mathrm{dB}}(d_0)\right] - 10\beta\log_{10}\left(\frac{d}{d_0}\right) + \epsilon,
\qquad \epsilon \sim \mathcal{N}(0,\sigma^2).$$
In dB the model is a straight line in $x = 10\log_{10}(d/d_0)$ with slope $-\beta$, so a least-squares line through
measurements estimates $\beta$, and the RMS scatter around the line estimates the shadow standard deviation $\sigma$.

Below we "measure" $n$ random locations, then fit the model. Try few points and large $\sigma$: how well can you pin down $\beta$?

In [ ]:
def show_fit(beta=3.5, sigma=8.0, n_points=200, seed=1):
    d0, p0 = 100.0, -60.0                      # reference: E[P_RX(100 m)] = -60 dBm
    g = np.random.default_rng(seed)
    d = 10 ** g.uniform(np.log10(50), np.log10(3000), n_points)
    p = pl.shadowed_rx_dbm(d, p0, beta, sigma, d0=d0, rng=g)
    p0_hat, beta_hat, sigma_hat = pl.fit_log_distance(d, p, d0=d0)

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw=dict(width_ratios=[2, 1]))
    dd = np.logspace(np.log10(50), np.log10(3000), 100)
    ax.semilogx(d, p, ".", color="tab:gray", alpha=0.6, label="simulated measurements")
    ax.semilogx(dd, pl.log_distance_rx_dbm(dd, p0, beta, d0), "k-", lw=2, label=fr"true mean, $\beta$={beta:.2f}")
    fit = pl.log_distance_rx_dbm(dd, p0_hat, beta_hat, d0)
    ax.semilogx(dd, fit, "--", color="tab:blue", lw=2, label=fr"LS fit, $\hat\beta$={beta_hat:.2f}")
    ax.fill_between(dd, fit - sigma_hat, fit + sigma_hat, color="tab:blue", alpha=0.15,
                    label=fr"fit $\pm\hat\sigma$ ($\hat\sigma$={sigma_hat:.1f} dB)")
    ax.set_xlabel("distance d (m)"); ax.set_ylabel(r"$P_{\mathrm{RX}}$ (dBm)")
    ax.set_title("Log-distance path loss + lognormal shadowing")
    ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=9)

    resid = p - pl.log_distance_rx_dbm(d, p0_hat, beta_hat, d0)
    ax2.hist(resid, bins=30, density=True, color="tab:gray", alpha=0.6, label="fit residuals")
    e = np.linspace(-4 * sigma, 4 * sigma, 200)
    ax2.plot(e, np.exp(-e**2 / (2 * sigma**2)) / np.sqrt(2 * np.pi * sigma**2), "k-", label=fr"$\mathcal{{N}}(0,{sigma:g}^2)$")
    ax2.set_xlabel(r"residual $\epsilon$ (dB)"); ax2.set_ylabel("density")
    ax2.set_title("Shadowing is Gaussian in dB"); ax2.legend(fontsize=9)
    plt.tight_layout(); plt.show()

interact(show_fit,
         beta=FloatSlider(value=3.5, min=2.0, max=5.0, step=0.1),
         sigma=FloatSlider(value=8.0, min=0.0, max=12.0, step=0.5),
         n_points=IntSlider(value=200, min=10, max=2000, step=10),
         seed=IntSlider(value=1, min=0, max=20));

How reliable is the fit? Repeat the "measurement campaign" many times and look at the spread of $\hat\beta$.
With $x_k = 10\log_{10}(d_k/d_0)$, least-squares theory gives $\mathrm{std}(\hat\beta) = \sigma / \sqrt{\sum_k (x_k-\bar x)^2}$.

In [ ]:
beta, sigma, d0 = 3.5, 8.0, 100.0
print(f"{'n points':>9} {'mean beta_hat':>14} {'std (sim)':>10} {'std (theory)':>13} {'mean sigma_hat':>15}")
for n in [10, 30, 100, 300, 1000]:
    est = []
    for trial in range(300):
        d = 10 ** rng.uniform(np.log10(50), np.log10(3000), n)
        est.append(pl.fit_log_distance(d, pl.shadowed_rx_dbm(d, -60, beta, sigma, d0, rng=rng), d0))
    est = np.array(est)
    # theory for the expected spread of x for log-uniform distances
    x_std = 10 * (np.log10(3000) - np.log10(50)) / np.sqrt(12)
    print(f"{n:>9} {est[:, 1].mean():14.2f} {est[:, 1].std():10.3f} {sigma / (x_std * np.sqrt(n)):13.3f} {est[:, 2].mean():15.2f}")

## 2. Co-channel interference: downlink SIR vs. $N$ and $\beta$
With equal transmit powers and $P_{\mathrm{RX}} \propto d^{-\beta}$, a mobile at distance $R$ from its base station
(worst case: the cell edge) sees
$$\Lambda = \frac{S}{\sum_{i\in i_0} I_i} = \frac{R^{-\beta}}{\sum_{i\in i_0} D_i^{-\beta}} .$$
The lecture gives two approximations for the six first-tier interferers ($i_0 = 6$), with $Q = D/R = \sqrt{3N}$:

* **Assumption #1** (all $D_i = D$): $\ \Lambda = Q^\beta / i_0$
* **Assumption #2** (two each at $D-R$, $D$, $D+R$): $\ \Lambda = \dfrac{1}{2}\,\dfrac{1}{(Q-1)^{-\beta} + Q^{-\beta} + (Q+1)^{-\beta}}$

"Why assume anything when you can simulate?" — `pl.worst_case_sir_geometric` places the co-channel base stations on
the exact hexagonal lattice (via `ece512.cells`) and searches the whole boundary of the home cell for the minimum $\Lambda$.

In [ ]:
# AMPS example from the slides: Lambda >= 18 dB, beta = 4, i0 = 6
print(f"AMPS: N = {pl.cluster_size_for_sir(18, 4):.2f} (numerically)  ->  choose N = 7")
print(f"Assumption #2, beta = 3.5: " + ", ".join(
    f"N={n}: {linear_to_db(pl.sir_assumption2(n, 3.5)):.2f} dB" for n in (7, 4, 3)))

# (i, j) for each allowed cluster size
IJ = {}
for i in range(6):
    for j in range(i + 1):
        n = cells.cluster_size(i, j)
        if 0 < n <= 27 and n not in IJ:
            IJ[n] = (i, j)
N_ALLOWED = sorted(IJ)
print("allowed N:", N_ALLOWED)

In [ ]:
def show_sir_vs_n(beta=3.5, sir_th_db=18.0):
    n = np.array(N_ALLOWED)
    a1 = linear_to_db(pl.sir_equal_distance(n, beta))
    a2 = linear_to_db(pl.sir_assumption2(n, beta))
    geo1 = np.array([linear_to_db(pl.worst_case_sir_geometric(*IJ[k], 1.0, beta)) for k in n])
    geo2 = np.array([linear_to_db(pl.worst_case_sir_geometric(*IJ[k], 1.0, beta, n_tiers=2)) for k in n])

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(n, a1, "o--", color="tab:blue", label=r"Assumption #1: $Q^\beta/6$")
    ax.plot(n, a2, "s--", color="tab:orange", label=r"Assumption #2: $D-R,\ D,\ D+R$")
    ax.plot(n, geo1, "^-", color="tab:green", label="exact hex geometry, 1st tier (worst point)")
    ax.plot(n, geo2, "v:", color="tab:purple", label="exact hex geometry, 2 tiers")
    ax.axhline(sir_th_db, color="k", lw=1)
    ax.text(n[0], sir_th_db + 0.4, r"$\Lambda_{\mathrm{TH}}$", ha="left", va="bottom")
    ax.set_xticks(n)
    ax.set_xlabel("cluster size N"); ax.set_ylabel(r"worst-case $\Lambda$ (dB)")
    ax.set_title(fr"Downlink SIR at the cell edge, $\beta$ = {beta:.1f}")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=9, loc="lower right"); plt.show()

    print(f"Smallest allowed N meeting Lambda_TH = {sir_th_db:.0f} dB:")
    for name, v in [("Assumption #1", a1), ("Assumption #2", a2), ("exact, 1st tier", geo1), ("exact, 2 tiers", geo2)]:
        ok = n[v >= sir_th_db]
        print(f"  {name:17s} N = {ok[0] if ok.size else '> 28'}")

interact(show_sir_vs_n,
         beta=FloatSlider(value=3.5, min=2.0, max=5.0, step=0.1),
         sir_th_db=FloatSlider(value=18.0, min=5.0, max=30.0, step=0.5));

Assumption #1 is optimistic (it ignores that the edge mobile is *closer* to some interferers); Assumption #2 is
slightly pessimistic, because in the real hexagonal layout the mobile cannot be $D-R$ from two interferers at once.
Adding the second tier costs only about 1 dB or less for $\beta \ge 3$.

## 3. Where in the cell is the SIR worst?
The map below evaluates $\Lambda(\mathbf{x}) = |\mathbf{x}|^{-\beta} / \sum_i |\mathbf{x}-\mathbf{c}_i|^{-\beta}$ at every point of the home cell
(two tiers of co-channel base stations $\mathbf{c}_i$). Choose the sectoring to see how ideal sector antennas
(unit gain inside the sector, zero outside) remove interferers. The red × marks the worst point on the map.

In [ ]:
def sir_grid(i, j, beta, n_sectors, n=161):
    R = 1.0
    xs = np.linspace(-R, R, n)
    X, Y = np.meshgrid(xs, xs)
    Z = X + 1j * Y
    v = hex_vertices(0j, R)
    inside = Path(np.column_stack([v.real, v.imag])).contains_points(
        np.column_stack([X.ravel(), Y.ravel()])).reshape(X.shape)
    sir = linear_to_db(pl.sir_map(Z, i, j, R, beta, n_sectors=n_sectors, n_tiers=2))
    return X, Y, np.where(inside & (np.abs(Z) > 1e-9), sir, np.nan)

def show_sir_map(N=7, beta=3.5, sectoring="omni"):
    i, j = IJ[N]
    ns = {"omni": 1, "120°": 3, "60°": 6}[sectoring]
    X, Y, S = sir_grid(i, j, beta, ns)
    fig, ax = plt.subplots(figsize=(6.6, 5.6))
    lo = np.floor(np.nanmin(S))
    pc = ax.pcolormesh(X, Y, S, shading="auto", cmap="viridis", vmin=lo, vmax=lo + 24)
    fig.colorbar(pc, ax=ax, label=r"$\Lambda$ (dB)", extend="max")
    cs = ax.contour(X, Y, S, levels=np.arange(lo + 2, lo + 23, 4), colors="w", linewidths=0.6)
    ax.clabel(cs, fmt="%d dB", fontsize=8)
    draw_cell(ax, 0j, 1.0, color="k", linewidth=1.5)
    if ns > 1:
        for k in range(ns):
            e = 1.0 * np.exp(1j * 2 * np.pi * k / ns)
            ax.plot([0, e.real], [0, e.imag], "k-", lw=1)
    k = np.nanargmin(S)
    ax.plot(X.flat[k], Y.flat[k], "x", color="tab:red", ms=12, mew=2.5, clip_on=False)
    ax.plot(0, 0, "k^", ms=9)
    ax.set_aspect("equal"); ax.set_xlabel("x / R"); ax.set_ylabel("y / R")
    ax.set_title(fr"N = {N} (i={i}, j={j}), $\beta$ = {beta:.1f}, {sectoring}: min $\Lambda$ = {np.nanmin(S):.1f} dB")
    plt.show()
    q = np.sqrt(3 * N)
    approx = {1: pl.sir_assumption2(N, beta), 3: pl.sir_sector120(N, beta), 6: pl.sir_sector60(N, beta)}[ns]
    print(f"Q = D/R = {q:.2f}; lecture worst-case formula for this sectoring: {linear_to_db(approx):.1f} dB")

interact(show_sir_map,
         N=Dropdown(options=[3, 4, 7, 9, 12, 13], value=7),
         beta=FloatSlider(value=3.5, min=2.0, max=5.0, step=0.1),
         sectoring=Dropdown(options=["omni", "120°", "60°"], value="omni"));

Without sectoring the worst locations are the cell corners (farthest from the serving base station).
With sectors the picture becomes asymmetric: only the co-channel sectors that point *toward* the home cell contribute.

## 4. Cell sectoring
Directional sector antennas reduce the number of first-tier interferers from $i_0 = 6$ (omni) to $i_0 = 2$ ($120^\circ$)
or $i_0 = 1$ ($60^\circ$). The lecture's worst-case approximations are
$$\Lambda_{120^\circ} = \frac{1}{(Q+0.7)^{-\beta} + Q^{-\beta}}, \qquad
\Lambda_{60^\circ} = (Q+0.7)^{\beta}.$$
Solid lines: exact geometry (worst point of the cell, first tier, ideal sectors); dashed: the lecture formulas.

In [ ]:
beta = 3.5
n = np.array([3, 4, 7, 9, 12, 13, 19])
formulas = {"omni": pl.sir_assumption2, "120°": pl.sir_sector120, "60°": pl.sir_sector60}
colors = {"omni": "tab:blue", "120°": "tab:orange", "60°": "tab:green"}
fig, ax = plt.subplots(figsize=(8, 4.5))
table = {}
for name, ns in [("omni", 1), ("120°", 3), ("60°", 6)]:
    geo = np.array([linear_to_db(pl.worst_case_sir_geometric(*IJ[k], 1.0, beta, n_sectors=ns)) for k in n])
    frm = linear_to_db(formulas[name](n, beta))
    table[name] = (frm, geo)
    ax.plot(n, geo, "o-", color=colors[name], label=f"{name}: exact geometry")
    ax.plot(n, frm, "--", color=colors[name], alpha=0.8, label=f"{name}: lecture formula")
ax.set_xticks(n); ax.set_xlabel("cluster size N"); ax.set_ylabel(r"worst-case $\Lambda$ (dB)")
ax.set_title(fr"Sectoring improves SIR ($\beta$ = {beta})"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, ncol=3, loc="lower right"); plt.show()

print(f"{'N':>3} | {'omni':^13} | {'120°':^13} | {'60°':^13}   (formula / exact, dB)")
for k, nn in enumerate(n):
    print(f"{nn:>3} | " + " | ".join(f"{table[s][0][k]:5.1f} / {table[s][1][k]:5.1f}" for s in table))

The formulas reproduce the slide values ($21.1$, $17.1$, $15.0$ dB for $120^\circ$ at $N = 7, 4, 3$).
The trade-off: every sector needs its own channel set, so the channels per sector (trunking efficiency, Week 3) drop,
and handoffs between sectors increase. A common design is $N = 7$ omni → $N = 4$ with $120^\circ$ sectors at similar SIR.

## 5. Thermal noise and noise figure
The input noise power of a receiver with noise bandwidth $B_w$ and noise figure $F$ is
$$N = k T_0 B_w F, \qquad N_{\mathrm{dBm}} = -174\ \mathrm{dBm/Hz} + 10\log_{10} B_w + F_{\mathrm{dB}} \quad (T_0 = 290\ \mathrm{K}).$$
The **sensitivity** is the smallest power that still achieves the required SNR (or $E_c/N_0$):
$S_{\mathrm{rx}} = kT_0B_w + F + \Gamma_{\min}$ (all in dB).

In [ ]:
print(f"N_0 = kT_0 = {pl.noise_psd_dbm_hz(290):.2f} dBm/Hz at 290 K;  {pl.noise_psd_dbm_hz(293):.2f} dBm/Hz at 293 K\n")
systems = [("GSM", 200e3), ("NB-IoT", 180e3), ("WCDMA", 3.84e6), ("LTE 20 MHz", 18e6),
           ("Wi-Fi 80 MHz", 80e6), ("5G NR FR2 400 MHz", 400e6), ("28 GHz testbed", 1e9)]
print(f"{'system':>20} {'B_w':>10} {'kTB (dBm)':>10} {'+F=7 dB':>9}")
for name, b in systems:
    print(f"{name:>20} {b/1e6:8.2f} MHz {pl.thermal_noise_dbm(b):10.1f} {pl.thermal_noise_dbm(b) + 7:9.1f}")

b = np.logspace(3, 10, 200)
fig, ax = plt.subplots(figsize=(8, 4))
for f_db, c in [(0, "k"), (3, "tab:blue"), (6, "tab:orange"), (10, "tab:green")]:
    ax.semilogx(b, pl.thermal_noise_dbm(b) + f_db, color=c, label=f"F = {f_db} dB")
for name, bw in systems[::2]:
    ax.plot(bw, pl.thermal_noise_dbm(bw), "k.", ms=8)
    ax.annotate(name, (bw, pl.thermal_noise_dbm(bw)), textcoords="offset points", xytext=(4, -12), fontsize=8)
ax.set_xlabel(r"noise bandwidth $B_w$ (Hz)"); ax.set_ylabel("noise floor (dBm)")
ax.set_title("Noise floor rises 10 dB per decade of bandwidth")
ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()

**Cascaded stages.** For a chain of stages with noise factors $F_k$ and gains $G_k$ (linear), Friis' noise formula gives
$F = F_1 + \frac{F_2-1}{G_1} + \frac{F_3-1}{G_1G_2} + \cdots$ — the first stage dominates when its gain is high,
which is why a low-noise amplifier (LNA) goes right after the antenna.

In [ ]:
def show_cascade(lna_nf_db=2.0, lna_gain_db=20.0, radio_nf_db=8.0):
    ftot = pl.cascade_noise_figure_db([lna_nf_db, radio_nf_db], [lna_gain_db, 0])
    frev = pl.cascade_noise_figure_db([radio_nf_db, lna_nf_db], [0, lna_gain_db])
    print(f"LNA ({lna_nf_db} dB NF, {lna_gain_db} dB gain) -> radio ({radio_nf_db} dB NF):  F = {ftot:.2f} dB")
    print(f"radio first, then LNA:                                F = {frev:.2f} dB")
    print(f"naive sum of the noise figures:                       F = {lna_nf_db + radio_nf_db:.2f} dB")

interact(show_cascade,
         lna_nf_db=FloatSlider(value=3.2, min=0.5, max=10, step=0.1),
         lna_gain_db=FloatSlider(value=24, min=0, max=40, step=1),
         radio_nf_db=FloatSlider(value=8, min=0, max=15, step=0.5));

## 6. Link budget calculator: the 28 GHz mmWave testbed
The lecture's case study links two USRPs through SpaceK 28 GHz down-converters and 10 dB antennas across a
33 ft × 36 ft grid. Signal and noise are both referred to the **SpaceK down-converter input**:
$$P_{\mathrm{RX}} = P_{\mathrm{TX}} + G_{\mathrm{TX}} + G_{\mathrm{RX}} - L_C - L_{\mathrm{FS}} \;\ge\; P_{\min} = kTB + F_{\mathrm{cascade}} + \mathrm{SNR},$$
$$F_{\mathrm{cascade}} = F_{\mathrm{SK}} + \frac{F_{\mathrm{USRP}}-1}{G_{\mathrm{SK}}}\ \ (\text{linear}),\qquad
L_{\mathrm{FS}} = 20\log_{10}(R_{\mathrm{km}}) + 20\log_{10}(f_{\mathrm{GHz}}) + 92.45 .$$
The SpaceK gain $G_{\mathrm{SK}}$ amplifies signal *and* noise, so it enters only through the cascaded noise figure.
The defaults reproduce the slides: $F_{\mathrm{cascade}} = 3.24$ dB, $P_{\min} = -60.69$ dBm, $L_{\mathrm{FS}} = 84.86$ dB,
$P_{\mathrm{RX}} = -37.86$ dBm, link margin $22.83$ dB, and a minimum antenna gain of $\approx -1.4$ dB per end.
Move the sliders to redesign the link.

In [ ]:
def link_budget(ptx_dbm=30.0, g_ant_db=10.0, l_c_db=3.0, dist_m=14.9, f_ghz=28.0, snr_db=20.0,
                g_sk_db=24.0, nf_sk_db=3.2, nf_usrp_db=8.0, bw_mhz=1000.0, temp_k=293.0):
    l_fs = pl.fspl_db_practical(dist_m, f_ghz * 1e9)   # 20log10(R_km) + 20log10(f_GHz) + 92.45
    floor = pl.thermal_noise_dbm(bw_mhz * 1e6, temp_k)
    f_cas = pl.cascade_noise_figure_db([nf_sk_db, nf_usrp_db], [g_sk_db, 0.0])
    p_min = pl.sensitivity_dbm(bw_mhz * 1e6, f_cas, snr_db, temp_k)
    steps = [("P_TX", ptx_dbm), ("+G_TX", g_ant_db), ("-L_C", -l_c_db), ("-L_FS", -l_fs), ("+G_RX", g_ant_db)]
    p_rx = pl.received_power_dbm(ptx_dbm, [g_ant_db, g_ant_db], [l_c_db, l_fs])
    g_min = (p_min - pl.received_power_dbm(ptx_dbm, [], [l_c_db, l_fs])) / 2
    d_max = dist_m * 10 ** ((p_rx - p_min) / 20)       # free space: 20 dB/decade

    levels = np.cumsum([v for _, v in steps])
    prev = np.concatenate([[0], levels[:-1]])
    x = np.arange(len(steps))
    fig, ax = plt.subplots(figsize=(9, 4.3))
    for k, (name, v) in enumerate(steps):
        if k == 0:          # starting level, not a gain: no bar
            ax.plot(k, v, "ks", ms=7); ax.text(k, v + 2.5, f"{v:.2f} dBm", ha="center", fontsize=9)
            continue
        lo, hi = sorted((prev[k], levels[k]))
        ax.bar(k, hi - lo, bottom=lo, width=0.6, color="tab:green" if v >= 0 else "tab:red", alpha=0.8)
        ax.text(k, hi + 2.5, f"{v:+.2f}", ha="center", fontsize=9)
    ax.step(np.append(x, x[-1] + 1) - 0.5, np.append(levels, levels[-1]), where="post", color="k", lw=1)
    ax.axhline(p_min, color="tab:blue", ls="--")
    ax.text(len(steps) - 0.5, p_min - 3, fr"$P_{{\min}}$ = kTB + $F_{{\mathrm{{cascade}}}}$ + SNR = {p_min:.2f} dBm",
            ha="right", va="top", color="tab:blue", fontsize=9)
    ax.axhline(floor, color="gray", ls=":"); ax.text(-0.4, floor + 1.5, f"kTB = {floor:.2f} dBm", color="gray", fontsize=9)
    ax.annotate("", xy=(len(steps) - 0.5, p_rx), xytext=(len(steps) - 0.5, p_min),
                arrowprops=dict(arrowstyle="<->", color="k"))
    ax.text(len(steps) - 0.45, (p_rx + p_min) / 2, f"margin\n{p_rx - p_min:.2f} dB", va="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([s for s, _ in steps]); ax.set_xlim(-0.6, len(steps) + 0.6)
    ax.set_ylabel("power level at SpaceK input (dBm)"); ax.grid(True, axis="y", alpha=0.3)
    ax.set_ylim(min(floor, p_min, levels.min()) - 10, max(levels.max(), 0) + 10)
    ax.set_title(fr"Link budget: $P_{{\mathrm{{RX}}}}$ = {p_rx:.2f} dBm at {dist_m:.1f} m, {f_ghz:g} GHz")
    plt.tight_layout(); plt.show()
    print(f"free-space path loss L_FS          = {l_fs:7.2f} dB")
    print(f"noise floor 10log10(kTB)           = {floor:7.2f} dBm")
    print(f"cascaded noise figure F_cascade    = {f_cas:7.2f} dB")
    print(f"minimum received power P_min       = {p_min:7.2f} dBm")
    print(f"received power P_RX                = {p_rx:7.2f} dBm")
    print(f"link margin P_RX - P_min           = {p_rx - p_min:7.2f} dB")
    print(f"minimum antenna gain (each end)    = {g_min:7.2f} dB")
    print(f"max range at zero margin (FS)      = {d_max:7.1f} m")

R_grid = np.hypot(33, 36) * 0.3048
print(f"grid diagonal R = sqrt(33^2 + 36^2) ft = {np.hypot(33, 36):.1f} ft = {R_grid:.1f} m")
interact(link_budget,
         ptx_dbm=FloatSlider(value=30, min=-10, max=40, step=1),
         g_ant_db=FloatSlider(value=10, min=-15, max=30, step=0.5),
         l_c_db=FloatSlider(value=3, min=0, max=15, step=0.5),
         dist_m=FloatSlider(value=14.9, min=1, max=500, step=0.1),
         f_ghz=FloatSlider(value=28, min=0.7, max=100, step=0.1),
         snr_db=FloatSlider(value=20, min=0, max=40, step=1),
         g_sk_db=FloatSlider(value=24, min=0, max=40, step=1),
         nf_sk_db=FloatSlider(value=3.2, min=0, max=15, step=0.1),
         nf_usrp_db=FloatSlider(value=8, min=0, max=15, step=0.1),
         bw_mhz=FloatSlider(value=1000, min=1, max=2000, step=1),
         temp_k=FloatSlider(value=293, min=250, max=350, step=1));

**Why $G_{\mathrm{SK}}$ belongs in the noise-figure cascade.** The SpaceK down-converter amplifies whatever arrives at
its input — the wanted signal *and* the thermal noise — by the same 24 dB, so its gain alone does not change the SNR.
What it does change is how much the noisy USRP behind it matters: the USRP's excess noise is divided by $G_{\mathrm{SK}}$,
so $F_{\mathrm{cascade}} \approx F_{\mathrm{SK}}$. Signal and noise must be referred to the *same* point (a reference plane).

*A common mistake* is to add $G_{\mathrm{SK}}$ to the signal budget while comparing against a noise floor that was never
amplified, and to add the noise figures in dB. Try `g_sk_db = 0` above: with no pre-amplifier the USRP's 8 dB noise figure
dominates. The comparison below shows how much the naive method overstates the margin.

In [ ]:
l_fs = pl.fspl_db_practical(14.9, 28e9)
floor = pl.thermal_noise_dbm(1e9, 293)
f_cas = pl.cascade_noise_figure_db([3.2, 8.0], [24.0, 0.0])
correct = pl.received_power_dbm(30, [10, 10], [3, l_fs]) - pl.sensitivity_dbm(1e9, f_cas, 20, 293)
naive = pl.received_power_dbm(30, [10, 10, 24], [3, l_fs]) - (floor + 3.2 + 8.0 + 20)
print(f"consistent (cascade NF {f_cas:.2f} dB, both at SpaceK input): margin = {correct:5.2f} dB")
print(f"naive (G_SK in signal only, NFs summed to 11.2 dB):       margin = {naive:5.2f} dB  "
      f"-> overstated by {naive - correct:.2f} dB")

## 7. Shadow margin, handoff gain, and coverage
### 7a. Edge and area outage
At the cell edge $P_{\mathrm{RX}|\mathrm{dB}}(R) \sim \mathcal{N}(\mu(R), \sigma^2)$. Designing $\mu(R)$ to sit a
**shadow margin** $M_{\mathrm{SHAD}} = \mu(R) - P_{\mathrm{TH}}$ above threshold gives
$$P_{\mathrm{EDGE}} = Q\left(\frac{M_{\mathrm{SHAD}}}{\sigma}\right), \qquad
P_{\mathrm{AREA}} = Q(X) - \exp\left(XY + \frac{Y^2}{2}\right) Q(X+Y),\quad X = \frac{M_{\mathrm{SHAD}}}{\sigma},\ Y = \frac{2\sigma\ln(10)}{10\beta}.$$
Markers: Monte Carlo with users uniform over a circular cell.

In [ ]:
print(f"Slide example: P_EDGE = 0.1, sigma = 8 dB -> M_SHAD = 8 Q^-1(0.1) = 8 x {pl.qfuncinv(0.1):.4f} = {pl.shadow_margin_db(0.1, 8):.2f} dB")
print(f"  (the same margin gives area outage P_AREA = {pl.area_outage(pl.shadow_margin_db(0.1, 8), 8, 3.5):.3f} for beta = 3.5)\n")

m = np.linspace(-5, 25, 121); m_mc = np.arange(-4, 25, 3)
fig, ax = plt.subplots(figsize=(8, 4.5))
for sigma, c in [(4, "tab:blue"), (8, "tab:orange"), (12, "tab:green")]:
    ax.semilogy(m, pl.edge_outage(m, sigma), "--", color=c, label=fr"edge, $\sigma$ = {sigma} dB")
    ax.semilogy(m, pl.area_outage(m, sigma, 3.5), "-", color=c, label=fr"area, $\sigma$ = {sigma} dB")
    mc = [pl.simulate_area_outage(mm, sigma, 3.5, n_users=40_000, rng=rng) for mm in m_mc]
    ax.semilogy(m_mc, np.maximum(mc, 1e-6), "o", color=c, ms=5, mfc="none")
ax.plot([], [], "ko", mfc="none", label="area, Monte Carlo")
ax.set_ylim(1e-3, 1); ax.set_xlabel(r"shadow margin $M_{\mathrm{SHAD}}$ (dB)"); ax.set_ylabel("outage probability")
ax.set_title(r"Edge vs. area outage ($\beta$ = 3.5)"); ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=2); plt.show()

In [ ]:
def show_margin(p_target=0.1, sigma=8.0, beta=3.5):
    m_edge = pl.shadow_margin_db(p_target, sigma)
    m_area = pl.area_outage_margin_db(p_target, sigma, beta)
    e = np.linspace(-4 * sigma, 4 * sigma, 400)
    pdf = np.exp(-e**2 / (2 * sigma**2)) / np.sqrt(2 * np.pi * sigma**2)
    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(e + m_edge, pdf, "k-")
    tail = e + m_edge < 0
    ax.fill_between((e + m_edge)[tail], pdf[tail], color="tab:red", alpha=0.4,
                    label=fr"$P_{{\mathrm{{EDGE}}}} = Q(M_{{\mathrm{{SHAD}}}}/\sigma)$ = {p_target:.3f}")
    ax.axvline(0, color="tab:red"); ax.text(0, pdf.max() * 1.02, r"$P_{\mathrm{TH}}$ ", color="tab:red", ha="right")
    ax.axvline(m_edge, color="k", ls=":"); ax.text(m_edge, pdf.max() * 1.02, r" $\mu(R)$")
    ax.set_xlabel(r"$P_{\mathrm{RX}|\mathrm{dB}}(R) - P_{\mathrm{TH}}$ (dB)"); ax.set_ylabel("pdf")
    ax.set_ylim(0, pdf.max() * 1.15); ax.legend(loc="upper right"); ax.set_title("Received power at the cell edge")
    plt.show()
    print(f"edge-outage design: M_SHAD = sigma Q^-1({p_target:g}) = {m_edge:.2f} dB  (area outage then {pl.area_outage(m_edge, sigma, beta):.4f})")
    print(f"area-outage design: M_SHAD = {m_area:.2f} dB gives P_AREA = {p_target:g}  -> saves {m_edge - m_area:.2f} dB")

interact(show_margin,
         p_target=FloatSlider(value=0.1, min=0.01, max=0.3, step=0.01),
         sigma=FloatSlider(value=8.0, min=2.0, max=12.0, step=0.5),
         beta=FloatSlider(value=3.5, min=2.0, max=5.0, step=0.1));

### 7b. Handoff gain
At the boundary a mobile can hand off to a neighbor whose link may not be shadowed (macrodiversity). With two
equidistant base stations and shadowing correlation $\rho$ between the links, outage requires **both** links below
threshold. The margin needed for the same edge outage drops; the difference is the handoff gain $G_{\mathrm{HO}}$.
(Here we model an ideal, instantaneous choice of the better base station — an upper bound closer to soft handoff.)

In [ ]:
def show_handoff(sigma=8.0, rho=0.5, p_target=0.1):
    m = np.linspace(-5, 25, 121); m_mc = np.arange(-4, 22, 3)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.semilogy(m, pl.edge_outage(m, sigma), "-", color="tab:blue", label="single cell: Q(M/σ)")
    for nbs, c in [(2, "tab:orange"), (3, "tab:green")]:
        ax.semilogy(m, pl.handoff_edge_outage(m, sigma, nbs, rho), "-", color=c, label=f"best of {nbs} BSs")
        mc = [pl.simulate_handoff_edge_outage(mm, sigma, nbs, rho, n_trials=20_000, rng=rng) for mm in m_mc]
        ax.semilogy(m_mc, np.maximum(mc, 1e-6), "o", color=c, mfc="none", ms=5)
    m1 = pl.shadow_margin_db(p_target, sigma); m2 = pl.handoff_margin_db(p_target, sigma, 2, rho)
    ax.axhline(p_target, color="gray", lw=0.8)
    ax.annotate("", xy=(m2, p_target), xytext=(m1, p_target), arrowprops=dict(arrowstyle="<->", color="k"))
    ax.text((m1 + m2) / 2, p_target * 1.25, fr"$G_{{\mathrm{{HO}}}}$ = {m1 - m2:.1f} dB", ha="center")
    ax.set_ylim(1e-3, 1); ax.set_xlabel("margin at cell edge (dB)"); ax.set_ylabel("edge outage probability")
    ax.set_title(fr"Handoff gain, $\sigma$ = {sigma} dB, $\rho$ = {rho}  (circles: Monte Carlo)")
    ax.grid(True, which="both", alpha=0.3); ax.legend(); plt.show()

interact(show_handoff,
         sigma=FloatSlider(value=8.0, min=2.0, max=12.0, step=0.5),
         rho=FloatSlider(value=0.5, min=0.0, max=0.95, step=0.05),
         p_target=FloatSlider(value=0.1, min=0.01, max=0.3, step=0.01));

### 7c. Revised link budget and radio coverage
$$L_{\mathrm{PATH,MAX}} = P_T + G_T + G_R - S_{\mathrm{RX}} - M_{\mathrm{SHAD}} - L_I + G_{\mathrm{HO}}, \qquad
\frac{N_1}{N_2} = 10^{-2(L_1 - L_2)/(10\beta)} .$$
A few dB of link budget change the number of cell sites needed to cover an area substantially
(slides: $\beta = 3.5$, $L_1 - L_2 = 2$ dB gives $N_2/N_1 = 1.30$, i.e. 30% more sites).

In [ ]:
# Example: a macro-cell budget (illustrative numbers) with the three adjustments
ptx, gt, gr = 43.0, 15.0, 0.0                    # dBm, dBi, dBi
srx = pl.sensitivity_dbm(5e6, 7.0, 5.0)          # 5 MHz, NF 7 dB, SNR_min 5 dB
m_shad = pl.shadow_margin_db(0.1, 8.0); l_i = 3.0; g_ho = pl.handoff_gain_db(0.1, 8.0, 2, 0.5)
l0 = pl.max_path_loss_db(ptx, gt, gr, srx)
l1 = pl.max_path_loss_db(ptx, gt, gr, srx, m_shad, l_i, g_ho)
print(f"S_RX = {srx:.1f} dBm;  M_SHAD = {m_shad:.2f} dB, L_I = {l_i:.1f} dB, G_HO = {g_ho:.2f} dB")
print(f"L_PATH,MAX: {l0:.1f} dB (no margins)  ->  {l1:.1f} dB (revised)")
# 1 km reference loss 128 dB, beta = 3.5 (typical urban macro)
print(f"cell radius: {pl.max_range(l0, 128, 3.5, 1.0):.2f} km  ->  {pl.max_range(l1, 128, 3.5, 1.0):.2f} km")
print(f"slide example: N1/N2 = {pl.cell_count_ratio(2, 3.5):.3f}, N2/N1 = {1 / pl.cell_count_ratio(2, 3.5):.2f}\n")

dl = np.linspace(0, 10, 101)
fig, ax = plt.subplots(figsize=(8, 4))
for b, c in [(2.0, "tab:blue"), (3.0, "tab:orange"), (3.5, "tab:green"), (4.0, "tab:purple")]:
    ax.plot(dl, 1 / pl.cell_count_ratio(dl, b), color=c, label=fr"$\beta$ = {b}")
ax.plot(2, 1 / pl.cell_count_ratio(2, 3.5), "ko"); ax.annotate("slide example (1.30)", (2, 1.3), xytext=(2, 3), ha="center", arrowprops=dict(arrowstyle="->"))
ax.set_xlabel(r"link budget advantage $L_1 - L_2$ (dB)"); ax.set_ylabel(r"$N_2 / N_1$ (extra sites needed)")
ax.set_title("Small link-budget differences cost many cell sites"); ax.grid(True, alpha=0.3); ax.legend(); plt.show()

## Try it yourself
* In Section 1, how many measurement locations do you need so that $\hat\beta$ is within $\pm 0.1$ of the truth
  (one standard deviation) when $\sigma = 8$ dB? What if the measurements span only 100–300 m?
* The AMPS design picked $N = 7$ with Assumption #1. Using the exact-geometry curve in Section 2 with $\beta = 3.5$,
  which $N$ would you choose for $\Lambda_{\mathrm{TH}} = 18$ dB, with and without $120^\circ$ sectoring?
* In the 28 GHz link budget, how far apart could the antennas be at zero margin? At 60 GHz? Which single parameter
  change buys the most range per dB?
* For $\sigma = 8$ dB and a 10% edge outage target, how does the handoff gain change as the shadowing correlation
  $\rho$ goes from 0 to 0.9? Why does correlation hurt macrodiversity?